# Tugas MLOps : Experiment Tracking dengan MLflow (Fashion-MNIST + CNN)


| # | Syarat tugas | Bagian |
|---|---|---|
| 1 | MLflow server lokal (SQLite + folder artifact) | 2 |
| 2 | Dataset citra kecil (subset Fashion-MNIST) | 3 |
| 3 | Baseline CNN, log params, metrik per epoch, model | 5 |
| 4 | ≥ 20 trial Optuna dengan nested runs | 6 |
| 5 | Konfigurasi terbaik × 3 seed, mean ± std | 7 |
| 6 | Ablation ≥ 2 komponen (augmentasi, dropout) | 8 |
| 7 | Register, alias `@champion`, load & prediksi via `models:/…@champion` | 9 |
| 8 | Versi dataset dengan DVC, hash di setiap run | 3 (hash), semua bagian (log) |



## 1. Setup & konfigurasi

In [2]:
import os, sys, time, copy, random, hashlib, subprocess, tempfile
import numpy as np, pandas as pd, torch, requests, yaml
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import mlflow, optuna
from mlflow import MlflowClient

os.environ["DVC_NO_ANALYTICS"] = "1"
optuna.logging.set_verbosity(optuna.logging.WARNING)

# konfigurasi 
AUTHOR      = "raihana adelia putri"
EXPERIMENT  = "fashion-mnist-cnn"
MODEL_NAME  = "fashion-cnn"
N_PER_CLASS = 1000            # 1000 gambar/kelas -> subset 10.000 gambar
EPOCHS      = 8
N_TRIALS    = 20              # syarat tugas: minimal 20
SEEDS       = [42, 43, 44]
CLASSES     = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
BASE_CFG    = {"filters": 16, "dropout": 0.3, "lr": 1e-3, "batch_size": 64,
               "epochs": EPOCHS, "augment": True, "seed": 42}

def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

## 2. MLflow Tracking Server: SQLite + folder artifact 

In [3]:
# Setara dengan perintah di terminal (slide 41):
#   mlflow server --backend-store-uri sqlite:///mlflow.db \
#       --default-artifact-root ./mlartifacts --host 127.0.0.1 --port 5000
def server_up():
    try:
        return requests.get("http://127.0.0.1:5000/health", timeout=2).status_code == 200
    except requests.RequestException:
        return False

if not server_up():
    server = subprocess.Popen(
        [sys.executable, "-m", "mlflow", "server",
         "--backend-store-uri", "sqlite:///mlflow.db",
         "--default-artifact-root", "./mlartifacts",
         "--host", "127.0.0.1", "--port", "5000"],
        stdout=open("mlflow_server.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if server_up():
            break
        time.sleep(1)
assert server_up(), "Server MLflow belum jalan, lihat mlflow_server.log"

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENT)
print("MLflow siap: http://127.0.0.1:5000")

MLflow siap: http://127.0.0.1:5000


## 3. Dataset citra + versioning DVC \

Subset Fashion-MNIST: 1000 gambar per kelas (10.000 total), dibagi stratified 60/20/20 menjadi train/val/test. Indeks split ikut disimpan di file data, jadi **split terkunci dan ikut terversi**. File dibuat sekali dan tidak diubah lagi, supaya hash DVC-nya stabil.

In [4]:
from torchvision.datasets import FashionMNIST

os.makedirs("data", exist_ok=True)
DATA_FILE = "data/fashion_subset.npz"

if not os.path.exists(DATA_FILE):
    raw = FashionMNIST("data_raw", train=True, download=True)
    x_all, y_all = raw.data.numpy(), raw.targets.numpy()
    rng = np.random.RandomState(42)
    idx = np.hstack([rng.choice(np.where(y_all == c)[0], N_PER_CLASS, replace=False) for c in range(10)])
    rng.shuffle(idx)
    x, y = x_all[idx], y_all[idx]
    i_tr, i_tmp = train_test_split(np.arange(len(y)), test_size=0.4, stratify=y, random_state=42)
    i_va, i_te = train_test_split(i_tmp, test_size=0.5, stratify=y[i_tmp], random_state=42)
    np.savez(DATA_FILE, x=x, y=y, i_tr=i_tr, i_va=i_va, i_te=i_te)

with np.load(DATA_FILE) as f:        # `with` menutup file, jadi dvc add tidak bentrok di Windows
    d = {k: f[k] for k in f.files}   # semua array dimuat ke memori
def get(split):
    i = d[split]
    return (d["x"][i].astype("float32") / 255.0)[:, None], d["y"][i]     # (N, 1, 28, 28), (N,)
Xtr, ytr = get("i_tr"); Xva, yva = get("i_va"); Xte, yte = get("i_te")
print("train/val/test:", Xtr.shape, Xva.shape, Xte.shape)

train/val/test: (6000, 1, 28, 28) (2000, 1, 28, 28) (2000, 1, 28, 28)


In [5]:
# Versioning data dengan DVC 
def dvc(*args):
    r = subprocess.run([sys.executable, "-m", "dvc", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"dvc {' '.join(args)} gagal:\n{r.stdout}\n{r.stderr}")
    return r.stdout

if not os.path.isdir(".dvc"):
    dvc("init", "-q", *([] if os.path.isdir(".git") else ["--no-scm"]))   # --no-scm bila bukan repo git
dvc("add", DATA_FILE)                                                    # buat data/fashion_subset.npz.dvc

DATA_MD5 = yaml.safe_load(open(DATA_FILE + ".dvc"))["outs"][0]["md5"]
print("DATA_MD5 (dari DVC):", DATA_MD5)

# Jika memakai git, commit pointer-nya agar versi data terkait dengan versi kode:
#   git add data/fashion_subset.npz.dvc data/.gitignore .dvc
#   git commit -m "data: fashion subset v1"
# Opsional, simpan data di remote:  dvc remote add -d storage <path/bucket>  &&  dvc push

DATA_MD5 (dari DVC): b0eb26a17118f4f5ae921d336fc3936c


## 4. Helper: model, training per epoch, dan logging bersama

`log_common()` dipanggil di **setiap** run (parent maupun child) sehingga hash data (`data_md5`), commit git, dan penulis selalu tercatat.

In [6]:
def build_cnn(filters=16, dropout=0.3):
    return nn.Sequential(
        nn.Conv2d(1, filters, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(filters, filters * 2, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Dropout(dropout), nn.Linear(filters * 2 * 7 * 7, 10))

def make_loader(X, y, bs, shuffle, seed):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y).long())
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=torch.Generator().manual_seed(seed))

def augment(xb):
    """Augmentasi sederhana: geser acak (maks 2 piksel) + flip horizontal acak per gambar."""
    dx, dy = torch.randint(-2, 3, (2,)).tolist()
    xb = torch.roll(xb, shifts=(dy, dx), dims=(2, 3))
    flip = torch.rand(len(xb)) < 0.5
    return torch.where(flip[:, None, None, None], xb.flip(3), xb)

def train_one_epoch(model, loader, optimizer, use_aug):
    model.train(); total = 0.0
    for xb, yb in loader:
        if use_aug:
            xb = augment(xb)
        optimizer.zero_grad()
        loss = nn.functional.cross_entropy(model(xb), yb)
        loss.backward(); optimizer.step()
        total += loss.item() * len(xb)
    return total / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    with torch.no_grad():
        out = torch.cat([model(xb) for xb, _ in loader]); y = torch.cat([yb for _, yb in loader])
    return nn.functional.cross_entropy(out, y).item(), f1_score(y.numpy(), out.argmax(1).numpy(), average="macro")

# ---- pencatatan konteks (code, data, environment) ----
def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return "no-git"
GIT_COMMIT = git_commit()
with open("requirements.txt", "w") as f:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f)

def log_common(cfg=None, stage=None, env=False):
    """Dipanggil di SETIAP run: params, hash data DVC, commit git, tag."""
    if cfg:
        mlflow.log_params(cfg)
    mlflow.log_param("data_md5", DATA_MD5)
    mlflow.set_tags({"author": AUTHOR, "git_commit": GIT_COMMIT, "dvc_file": DATA_FILE + ".dvc"})
    if stage:
        mlflow.set_tag("stage", stage)
    if env:
        mlflow.log_artifact("requirements.txt")

def fit(cfg, eval_test=False, log_ckpt=False, log_model=False):
    """Latih CNN dan catat metrik PER EPOCH ke run yang sedang aktif. Mengembalikan metrik terbaik."""
    set_seed(cfg["seed"])
    tr = make_loader(Xtr, ytr, cfg["batch_size"], True, cfg["seed"])
    va = make_loader(Xva, yva, 256, False, cfg["seed"])
    model = build_cnn(cfg["filters"], cfg["dropout"])
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    best_f1, best_state = -1.0, None
    for epoch in range(cfg["epochs"]):
        train_loss = train_one_epoch(model, tr, opt, cfg["augment"])
        val_loss, val_f1 = evaluate(model, va)
        mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss, "val_f1_macro": val_f1}, step=epoch)
        if val_f1 > best_f1:
            best_f1, best_state = val_f1, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)                      # bobot epoch terbaik
    mlflow.log_metric("best_val_f1_macro", best_f1)
    res = {"val_f1_macro": best_f1, "test_f1_macro": None}
    if eval_test:                                          # test set hanya untuk kandidat final
        res["test_f1_macro"] = evaluate(model, make_loader(Xte, yte, 256, False, cfg["seed"]))[1]
        mlflow.log_metric("test_f1_macro", res["test_f1_macro"])
    if log_ckpt:
        with tempfile.TemporaryDirectory() as tmp:
            torch.save(best_state, os.path.join(tmp, "best.pt"))
            mlflow.log_artifact(os.path.join(tmp, "best.pt"), artifact_path="checkpoints")
    if log_model:
        mlflow.pytorch.log_model(model, name="model", input_example=Xva[:3])
    return res

## 5. Baseline CNN 

In [7]:
with mlflow.start_run(run_name="baseline-cnn") as run:
    log_common(BASE_CFG, "baseline", env=True)
    BASELINE = fit(BASE_CFG, log_ckpt=True, log_model=True)
print(f"Baseline val F1-macro: {BASELINE['val_f1_macro']:.4f}  (run {run.info.run_id[:8]})")

🏃 View run baseline-cnn at: http://127.0.0.1:5000/#/experiments/1/runs/348a9dbd2d354788871fc3b9141d3a61
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Baseline val F1-macro: 0.8225  (run 348a9dbd)


## 6. Hyperparameter tuning: Optuna + nested runs 


Parent run = sweep, child run = satu trial. Yang dicari: `lr`, `batch_size`, `dropout`, `filters`. Augmentasi tetap aktif.

In [8]:
def objective(trial):
    cfg = {**BASE_CFG,
           "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
           "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
           "dropout": trial.suggest_float("dropout", 0.0, 0.5),
           "filters": trial.suggest_categorical("filters", [8, 16, 32])}
    with mlflow.start_run(run_name=f"trial-{trial.number:02d}", nested=True):
        log_common(cfg, "trial")
        res = fit(cfg)
    return res["val_f1_macro"]

with mlflow.start_run(run_name="optuna-sweep") as sweep:
    log_common({"n_trials": N_TRIALS, "sampler": "TPE(seed=42)", "epochs": EPOCHS, "augment": True}, "tuning", env=True)
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS)
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_val_f1_macro", study.best_value)
    SWEEP_ID = sweep.info.run_id

BEST_CFG = {**BASE_CFG, **study.best_params}
print(f"{len(study.trials)} trial selesai. Terbaik: val F1-macro = {study.best_value:.4f}")
print("Konfigurasi terbaik:", study.best_params)

🏃 View run trial-00 at: http://127.0.0.1:5000/#/experiments/1/runs/1e8f665d856a43f58d3fa05adcadfd7a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial-01 at: http://127.0.0.1:5000/#/experiments/1/runs/7f226c6075824b9ba88fcb096ec48a7a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial-02 at: http://127.0.0.1:5000/#/experiments/1/runs/1da69ebf919c4bfb9d03a669b5fffefd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial-03 at: http://127.0.0.1:5000/#/experiments/1/runs/2480c75966fd4fb0ba836d0657ce44e5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial-04 at: http://127.0.0.1:5000/#/experiments/1/runs/7f71fedd2e504c93b07785285c2bc22e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial-05 at: http://127.0.0.1:5000/#/experiments/1/runs/3d4df3d1336945758ed6510895970024
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trial-06 at: http://127.0.0.1

In [9]:
# Cari & bandingkan run secara programatik (slide 49)
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT],
    filter_string=f"tags.mlflow.parentRunId = '{SWEEP_ID}'",
    order_by=["metrics.best_val_f1_macro DESC"],
    max_results=5,
)
runs[["tags.mlflow.runName", "params.lr", "params.batch_size", "params.dropout", "params.filters",
      "metrics.best_val_f1_macro"]]

,tags.mlflow.runName,params.lr,params.batch_size,params.dropout,params.filters,metrics.best_val_f1_macro
0,trial-15,0.004356491847439371,128,0.29889062064824284,32,0.856696
1,trial-11,0.005042763897419045,64,0.425352801554068,32,0.855496
2,trial-17,0.002547291662732835,32,0.16190608639169216,32,0.855487
3,trial-14,0.003063830169141876,128,0.26330636543887453,32,0.852878
4,trial-10,0.005711213985204134,64,0.4007532910858571,32,0.850556


## 7. Konfigurasi terbaik × 3 seed → mean ± std 

Satu seed tidak cukup (slide 21). Konfigurasi terbaik dijalankan ulang dengan seed 42, 43, 44. Std memakai ddof=1 (sampel). Test set dipakai di sini karena ini kandidat final.

In [10]:
final_rows = []
with mlflow.start_run(run_name="best-config-3seeds") as parent:
    log_common({k: v for k, v in BEST_CFG.items() if k != "seed"} | {"seeds": str(SEEDS)}, "final", env=True)
    for seed in SEEDS:
        cfg = {**BEST_CFG, "seed": seed}
        with mlflow.start_run(run_name=f"final-seed{seed}", nested=True) as child:
            log_common(cfg, "final-seed")
            res = fit(cfg, eval_test=True, log_ckpt=True, log_model=True)
        final_rows.append({"seed": seed, "run_id": child.info.run_id, **res})
    df_final = pd.DataFrame(final_rows)
    for m in ["val_f1_macro", "test_f1_macro"]:
        mlflow.log_metric(f"final_{m}_mean", df_final[m].mean())
        mlflow.log_metric(f"final_{m}_std", df_final[m].std(ddof=1))

pm = lambda s: f"{s.mean():.4f} ± {s.std(ddof=1):.4f}"
print(df_final[["seed", "val_f1_macro", "test_f1_macro"]].round(4).to_string(index=False))
print(f"\nVal  F1-macro (mean ± std): {pm(df_final['val_f1_macro'])}")
print(f"Test F1-macro (mean ± std): {pm(df_final['test_f1_macro'])}")
print(f"Baseline val F1-macro (1 seed): {BASELINE['val_f1_macro']:.4f}")

🏃 View run final-seed42 at: http://127.0.0.1:5000/#/experiments/1/runs/06ff2d5233f14a09904036f6e5d1cdc8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run final-seed43 at: http://127.0.0.1:5000/#/experiments/1/runs/aa820a968e734c179921e98bae9e8db2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


🏃 View run final-seed44 at: http://127.0.0.1:5000/#/experiments/1/runs/2ff3125418db4fe684aa5377ffae44f2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run best-config-3seeds at: http://127.0.0.1:5000/#/experiments/1/runs/419884bd472641ecaae446a80fad13ee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
 seed  val_f1_macro  test_f1_macro
   42        0.8567         0.8454
   43        0.8330         0.8309
   44        0.8380         0.8287

Val  F1-macro (mean ± std): 0.8425 ± 0.0125
Test F1-macro (mean ± std): 0.8350 ± 0.0091
Baseline val F1-macro (1 seed): 0.8225


## 8. Ablation: augmentasi dan dropout 

Dari konfigurasi terbaik, satu komponen dimatikan per variasi, dengan 3 seed yang sama. Variasi `full` memakai hasil bagian 7 (tidak dilatih ulang). Tambah variasi lain cukup dengan menambah entri di `ABLATIONS`.


In [11]:
ABLATIONS = {"no_augment": {"augment": False}, "no_dropout": {"dropout": 0.0}}

abl_rows = [{"variant": "full", **r} for r in final_rows]
with mlflow.start_run(run_name="ablation-study") as abl_parent:
    log_common({"seeds": str(SEEDS), "variants": str(list(ABLATIONS))}, "ablation", env=True)
    for name, change in ABLATIONS.items():
        for seed in SEEDS:
            cfg = {**BEST_CFG, **change, "seed": seed}
            with mlflow.start_run(run_name=f"abl-{name}-seed{seed}", nested=True) as child:
                log_common(cfg, "ablation-run")
                mlflow.set_tag("ablation", name)
                res = fit(cfg, eval_test=True)
            abl_rows.append({"variant": name, "seed": seed, "run_id": child.info.run_id, **res})
    df_abl = pd.DataFrame(abl_rows)
    for name, g in df_abl.groupby("variant"):
        mlflow.log_metric(f"abl_{name}_val_f1_macro_mean", g["val_f1_macro"].mean())
        mlflow.log_metric(f"abl_{name}_test_f1_macro_mean", g["test_f1_macro"].mean())

summary = df_abl.groupby("variant").agg(
    val_f1_mean=("val_f1_macro", "mean"), val_f1_std=("val_f1_macro", lambda s: s.std(ddof=1)),
    test_f1_mean=("test_f1_macro", "mean"), test_f1_std=("test_f1_macro", lambda s: s.std(ddof=1)))
summary["delta_val_vs_full"] = summary["val_f1_mean"] - summary.loc["full", "val_f1_mean"]
summary.loc[["full", *ABLATIONS]].round(4)

🏃 View run abl-no_augment-seed42 at: http://127.0.0.1:5000/#/experiments/1/runs/815444116a034a73a7583a7c1398571a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run abl-no_augment-seed43 at: http://127.0.0.1:5000/#/experiments/1/runs/5276ca1f59e2445e865c458a7e63bd77
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run abl-no_augment-seed44 at: http://127.0.0.1:5000/#/experiments/1/runs/51176d0a5bea4fc19a243ff5f63e3119
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run abl-no_dropout-seed42 at: http://127.0.0.1:5000/#/experiments/1/runs/74a5d579cac74075b4318dfeee15ecbb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run abl-no_dropout-seed43 at: http://127.0.0.1:5000/#/experiments/1/runs/a938bed9f27c45d6a590d56baa7a4f2b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run abl-no_dropout-seed44 at: http://127.0.0.1:5000/#/experiments/1/runs/b1a46417bcda4cb9b3de3fcf98f86722
🧪 View experiment at: 

,val_f1_mean,val_f1_std,test_f1_mean,test_f1_std,delta_val_vs_full
variant,,,,,
full,0.8425,0.0125,0.8350,0.0091,0.0000
no_augment,0.8699,0.0031,0.8628,0.0016,0.0274
no_dropout,0.8360,0.0094,0.8289,0.0147,-0.0066


## 9. Model Registry: register → alias `@champion` → load & prediksi 

Model dipilih dari 3 run seed berdasarkan **val F1** (bukan test). Quality gate sederhana: rata-rata val F1 konfigurasi terbaik harus ≥ baseline. Jika lolos, alias `@champion` dipasang; aplikasi memuat model lewat alias, tanpa hard-code nomor versi.

In [12]:
client = MlflowClient()
best_seed = df_final.sort_values("val_f1_macro", ascending=False).iloc[0]

# 1. Registrasi model dari run terbaik
mv = mlflow.register_model(f"runs:/{best_seed['run_id']}/model", MODEL_NAME)

# 2. Anotasi & tandai sebagai challenger
passed = df_final["val_f1_macro"].mean() >= BASELINE["val_f1_macro"]
client.update_model_version(MODEL_NAME, mv.version, description=(
    f"CNN {BEST_CFG['filters']} filter, dropout={BEST_CFG['dropout']:.3f}, lr={BEST_CFG['lr']:.5f}, "
    f"seed={int(best_seed['seed'])}; data_md5={DATA_MD5}; git={GIT_COMMIT[:8]}"))
client.set_model_version_tag(MODEL_NAME, mv.version, "validation_status", "passed" if passed else "rejected")
client.set_model_version_tag(MODEL_NAME, mv.version, "data_md5", DATA_MD5)
client.set_registered_model_alias(MODEL_NAME, "challenger", mv.version)

# 3. Lolos quality gate -> promosikan
if passed:
    client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
alias = "champion" if passed else "challenger"
print(f"Versi {mv.version} -> @{alias}")

# 4. Aplikasi serving memuat via alias
model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{alias}")
logits = np.asarray(model.predict(Xte))
pred = logits.argmax(1)
print(f"\nPrediksi lewat models:/{MODEL_NAME}@{alias} pada {len(yte)} gambar test:")
print(f"  accuracy = {(pred == yte).mean():.4f} | F1-macro = {f1_score(yte, pred, average='macro'):.4f}")
for i in range(8):
    print(f"  gambar {i}: prediksi = {CLASSES[pred[i]]:<10} | label asli = {CLASSES[yte[i]]}")
# mlflow models serve -m "models:/fashion-cnn@champion" -p 8080     # REST endpoint

Successfully registered model 'fashion-cnn'.
2026/10/10 21:04:38 WARNING mlflow.tracking._model_registry.fluent: Run with id 06ff2d5233f14a09904036f6e5d1cdc8 has no artifacts at artifact path 'model', registering model based on models:/m-2cf0117445be4b90b12dda5eb47ff7d4 instead
2026/10/10 21:04:38 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fashion-cnn, version 1
Created version '1' of model 'fashion-cnn'.


Versi 1 -> @champion

Prediksi lewat models:/fashion-cnn@champion pada 2000 gambar test:
  accuracy = 0.8465 | F1-macro = 0.8454
  gambar 0: prediksi = Sandal     | label asli = Sandal
  gambar 1: prediksi = Pullover   | label asli = Pullover
  gambar 2: prediksi = Shirt      | label asli = Shirt
  gambar 3: prediksi = Pullover   | label asli = Pullover
  gambar 4: prediksi = Bag        | label asli = Bag
  gambar 5: prediksi = Pullover   | label asli = Coat
  gambar 6: prediksi = T-shirt    | label asli = T-shirt
  gambar 7: prediksi = Bag        | label asli = Bag
